# Online Retail Customer Analytics

## 02 — Data Cleaning and Preprocessing

**Module:** IT3091 – Machine Learning  
**Group:** 2026-DS-10  
**Track:** Guided Data Track  

**Primary Responsible Member:**  
Lakshitha Dilshan J.K.P. — IT24102621

**Responsibility:**  
Data Understanding, Exploratory Data Analysis, Data Cleaning, Preprocessing and Quality Assurance

**Group Review:**  
All four group members will review the preprocessing decisions before the processed datasets are merged into the `main` branch.

### Notebook Objectives

1. Load and verify the unchanged raw dataset.
2. Apply cleaning rules in a documented sequence.
3. Record before-and-after row counts for every decision.
4. Preserve cancellation and return records separately.
5. Create a valid customer-purchase dataset.
6. Create a merchandise basket-source dataset.
7. Perform reproducible quality checks.
8. Save processed datasets for downstream group tasks.

### Downstream Handover

- The cleaned customer-purchase dataset will be provided to **Nirmani K.H.D.T.** for feature engineering.
- The engineered customer-feature dataset will later be provided to **Palliyaguruge C.T.** for clustering.
- The cleaned merchandise basket dataset and final customer-segment assignments will be provided to **Senaratne P.A.R.T.** for association-rule mining and evaluation.

> The raw Excel file will never be overwritten. All derived datasets will be stored in `data/processed/`.

In [1]:
from pathlib import Path
import hashlib
import random
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Random seed: {RANDOM_SEED}")

Python version: 3.14.5
Pandas version: 3.0.5
NumPy version: 2.5.3
Random seed: 42


In [2]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

raw_data_path = (
    project_root
    / "data"
    / "raw"
    / "Online_Retail.xlsx"
)

processed_data_path = project_root / "data" / "processed"
outputs_table_path = project_root / "outputs" / "tables"

processed_data_path.mkdir(parents=True, exist_ok=True)
outputs_table_path.mkdir(parents=True, exist_ok=True)

print(f"Project root        : {project_root}")
print(f"Raw dataset path    : {raw_data_path}")
print(f"Processed-data path : {processed_data_path}")
print(f"Dataset exists      : {raw_data_path.exists()}")

assert raw_data_path.exists(), "Raw dataset was not found."

Project root        : /Users/lakshithadilshan/Documents/online-retail-customer-analytics
Raw dataset path    : /Users/lakshithadilshan/Documents/online-retail-customer-analytics/data/raw/Online_Retail.xlsx
Processed-data path : /Users/lakshithadilshan/Documents/online-retail-customer-analytics/data/processed
Dataset exists      : True


In [3]:
EXPECTED_SHA256 = (
    "43465a06f2ccf7c8b5bd2892bc7defb52"
    "f97487934fe93b16ae4c3936424676d"
)

def calculate_sha256(file_path, chunk_size=1024 * 1024):
    sha256_hash = hashlib.sha256()

    with open(file_path, "rb") as file:
        while chunk := file.read(chunk_size):
            sha256_hash.update(chunk)

    return sha256_hash.hexdigest()


actual_sha256 = calculate_sha256(raw_data_path)

print(f"Expected SHA-256  : {EXPECTED_SHA256}")
print(f"Actual SHA-256    : {actual_sha256}")
print(f"Fingerprint match : {actual_sha256 == EXPECTED_SHA256}")

assert actual_sha256 == EXPECTED_SHA256, (
    "Dataset fingerprint mismatch. Cleaning has been stopped."
)

Expected SHA-256  : 43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d
Actual SHA-256    : 43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d
Fingerprint match : True


In [4]:
raw_df = pd.read_excel(
    raw_data_path,
    sheet_name="Online Retail",
    engine="openpyxl"
)

working_df = raw_df.copy(deep=True)

print(f"Raw rows    : {len(raw_df):,}")
print(f"Raw columns : {raw_df.shape[1]}")
print(f"Working rows: {len(working_df):,}")

Raw rows    : 541,909
Raw columns : 8
Working rows: 541,909


In [5]:
expected_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
]

assert list(raw_df.columns) == expected_columns
assert raw_df.shape == (541_909, 8)
assert raw_df.duplicated().sum() == 5_268
assert raw_df["CustomerID"].isna().sum() == 135_080
assert raw_df["Description"].isna().sum() == 1_454

print("Raw dataset validation checks passed.")

Raw dataset validation checks passed.


In [6]:
preprocessing_audit = []

def record_cleaning_step(
    step,
    rule,
    before_rows,
    after_rows,
    reason
):
    preprocessing_audit.append({
        "Step": step,
        "Rule": rule,
        "Rows_Before": before_rows,
        "Rows_Removed": before_rows - after_rows,
        "Rows_After": after_rows,
        "Reason": reason
    })


print("Preprocessing audit log initialised.")

Preprocessing audit log initialised.


## 1. Exact Duplicate Removal

Exact duplicates are removed only from the working dataset. Repeated invoice numbers are retained because one invoice can legitimately contain multiple product lines.

One occurrence of every exact record is preserved, while the original raw dataframe remains unchanged.

In [7]:
before_rows = len(working_df)

deduplicated_df = (
    working_df
    .drop_duplicates()
    .copy()
    .reset_index(drop=True)
)

after_rows = len(deduplicated_df)

record_cleaning_step(
    step="C01",
    rule="Remove additional exact duplicate rows",
    before_rows=before_rows,
    after_rows=after_rows,
    reason=(
        "Exact duplicates could inflate quantities, revenue, "
        "frequency and association-rule support."
    )
)

print(f"Rows before       : {before_rows:,}")
print(f"Duplicates removed: {before_rows - after_rows:,}")
print(f"Rows after        : {after_rows:,}")
print(
    f"Duplicates remaining: "
    f"{deduplicated_df.duplicated().sum():,}"
)
print(f"Raw dataframe rows: {len(raw_df):,}")

Rows before       : 541,909
Duplicates removed: 5,268
Rows after        : 536,641
Duplicates remaining: 0
Raw dataframe rows: 541,909


In [8]:
clean_invoice_text = (
    deduplicated_df["InvoiceNo"]
    .astype(str)
    .str.strip()
    .str.upper()
)

clean_is_cancellation = clean_invoice_text.str.startswith(
    "C",
    na=False
)

clean_is_negative_quantity = (
    deduplicated_df["Quantity"] < 0
)

return_adjustment_mask = (
    clean_is_cancellation
    | clean_is_negative_quantity
)

returns_adjustments_df = (
    deduplicated_df.loc[return_adjustment_mask]
    .copy()
    .reset_index(drop=True)
)

returns_adjustments_df["RecordType"] = np.select(
    [
        returns_adjustments_df["InvoiceNo"]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.startswith("C", na=False),

        returns_adjustments_df["Quantity"] < 0
    ],
    [
        "Cancellation",
        "Negative quantity adjustment"
    ],
    default="Other"
)

print(
    f"Deduplicated cancellation rows: "
    f"{clean_is_cancellation.sum():,}"
)
print(
    f"Deduplicated negative-quantity rows: "
    f"{clean_is_negative_quantity.sum():,}"
)
print(
    f"Separated return/adjustment rows: "
    f"{len(returns_adjustments_df):,}"
)

display(
    returns_adjustments_df["RecordType"]
    .value_counts()
    .to_frame("Row Count")
)

Deduplicated cancellation rows: 9,251
Deduplicated negative-quantity rows: 10,587
Separated return/adjustment rows: 10,587


,Row Count
RecordType,
Cancellation,9251
Negative quantity adjustment,1336


In [9]:
# Step C02: Exclude C-prefixed cancellation invoices
before_rows = len(deduplicated_df)

non_cancellation_df = (
    deduplicated_df.loc[
        ~clean_is_cancellation
    ]
    .copy()
)

after_rows = len(non_cancellation_df)

record_cleaning_step(
    step="C02",
    rule="Exclude C-prefixed cancellation invoices",
    before_rows=before_rows,
    after_rows=after_rows,
    reason=(
        "Cancellation invoices do not represent positive "
        "customer purchases."
    )
)


# Step C03: Retain positive quantities
before_rows = len(non_cancellation_df)

positive_quantity_df = (
    non_cancellation_df.loc[
        non_cancellation_df["Quantity"] > 0
    ]
    .copy()
)

after_rows = len(positive_quantity_df)

record_cleaning_step(
    step="C03",
    rule="Retain rows with Quantity greater than zero",
    before_rows=before_rows,
    after_rows=after_rows,
    reason=(
        "Negative quantities represent returns or stock "
        "adjustments rather than positive purchases."
    )
)


# Step C04: Retain positive unit prices
before_rows = len(positive_quantity_df)

valid_purchase_df = (
    positive_quantity_df.loc[
        positive_quantity_df["UnitPrice"] > 0
    ]
    .copy()
    .reset_index(drop=True)
)

after_rows = len(valid_purchase_df)

record_cleaning_step(
    step="C04",
    rule="Retain rows with UnitPrice greater than zero",
    before_rows=before_rows,
    after_rows=after_rows,
    reason=(
        "Zero or negative prices cannot contribute valid "
        "positive purchase revenue."
    )
)

print(f"Deduplicated rows          : {len(deduplicated_df):,}")
print(f"After cancellation removal : {len(non_cancellation_df):,}")
print(f"After quantity validation  : {len(positive_quantity_df):,}")
print(f"Valid purchase rows        : {len(valid_purchase_df):,}")

Deduplicated rows          : 536,641
After cancellation removal : 527,390
After quantity validation  : 526,054
Valid purchase rows        : 524,878


In [10]:
preprocessing_audit_df = pd.DataFrame(
    preprocessing_audit
)

display(preprocessing_audit_df)

,Step,Rule,Rows_Before,Rows_Removed,Rows_After,Reason
0,C01,Remove additional exact duplicate rows,541909,5268,536641,"Exact duplicates could inflate quantities, rev..."
1,C02,Exclude C-prefixed cancellation invoices,536641,9251,527390,Cancellation invoices do not represent positiv...
2,C03,Retain rows with Quantity greater than zero,527390,1336,526054,Negative quantities represent returns or stock...
3,C04,Retain rows with UnitPrice greater than zero,526054,1176,524878,Zero or negative prices cannot contribute vali...


In [11]:
assert len(raw_df) == 541_909
assert len(deduplicated_df) == 536_641
assert deduplicated_df.duplicated().sum() == 0
assert len(returns_adjustments_df) == 10_587
assert len(valid_purchase_df) == 524_878

assert not (
    valid_purchase_df["InvoiceNo"]
    .astype(str)
    .str.upper()
    .str.startswith("C")
    .any()
)

assert (valid_purchase_df["Quantity"] > 0).all()
assert (valid_purchase_df["UnitPrice"] > 0).all()

print("Duplicate removal and purchase-validity checks passed.")

Duplicate removal and purchase-validity checks passed.


## 2. Customer Identification and Data-Type Standardisation

Customer segmentation requires a valid customer identifier. Missing customer IDs will not be imputed because no reliable information exists to reconstruct them.

Identifier and text variables are standardised for consistent grouping. The customer identifier is converted from floating point to integer only after missing identifiers are excluded.

In [12]:
standardised_purchase_df = valid_purchase_df.copy()

text_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Country"
]

for column in text_columns:
    standardised_purchase_df[column] = (
        standardised_purchase_df[column]
        .astype("string")
        .str.strip()
    )

standardised_purchase_df["StockCode"] = (
    standardised_purchase_df["StockCode"].str.upper()
)

standardised_purchase_df["InvoiceDate"] = pd.to_datetime(
    standardised_purchase_df["InvoiceDate"]
)

standardised_purchase_df["LineTotal"] = (
    standardised_purchase_df["Quantity"]
    * standardised_purchase_df["UnitPrice"]
)

print(standardised_purchase_df.dtypes)
print(
    "\nDuplicates created by standardisation:",
    standardised_purchase_df.duplicated().sum()
)
print(
    "Missing descriptions in valid purchases:",
    standardised_purchase_df["Description"].isna().sum()
)

InvoiceNo              string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                string
LineTotal             float64
dtype: object

Duplicates created by standardisation: 0
Missing descriptions in valid purchases: 0


In [13]:
anonymous_purchase_df = (
    standardised_purchase_df.loc[
        standardised_purchase_df["CustomerID"].isna()
    ]
    .copy()
    .reset_index(drop=True)
)

before_rows = len(standardised_purchase_df)

identified_purchase_df = (
    standardised_purchase_df.loc[
        standardised_purchase_df["CustomerID"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

identified_purchase_df["CustomerID"] = (
    identified_purchase_df["CustomerID"].astype("int64")
)

after_rows = len(identified_purchase_df)

record_cleaning_step(
    step="C05",
    rule="Retain valid purchases with an identified CustomerID",
    before_rows=before_rows,
    after_rows=after_rows,
    reason=(
        "Customer segmentation requires a reliable customer "
        "identifier; missing IDs cannot be safely imputed."
    )
)

print(f"Valid purchase rows       : {before_rows:,}")
print(f"Anonymous purchase rows   : {len(anonymous_purchase_df):,}")
print(f"Identified purchase rows  : {after_rows:,}")
print(
    f"Unique identified customers: "
    f"{identified_purchase_df['CustomerID'].nunique():,}"
)
print(
    f"CustomerID data type      : "
    f"{identified_purchase_df['CustomerID'].dtype}"
)

Valid purchase rows       : 524,878
Anonymous purchase rows   : 132,186
Identified purchase rows  : 392,692
Unique identified customers: 4,338
CustomerID data type      : int64


In [14]:
administrative_codes = {
    "AMAZONFEE",
    "B",
    "BANK CHARGES",
    "C2",
    "CRUK",
    "D",
    "DOT",
    "M",
    "PADS",
    "POST",
    "S",
    "TEST001",
    "TEST002"
}

identified_admin_mask = (
    identified_purchase_df["StockCode"]
    .isin(administrative_codes)
)

all_purchase_admin_mask = (
    standardised_purchase_df["StockCode"]
    .isin(administrative_codes)
)

administrative_purchase_df = (
    standardised_purchase_df.loc[
        all_purchase_admin_mask
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    f"Administrative rows among all valid purchases: "
    f"{all_purchase_admin_mask.sum():,}"
)
print(
    f"Administrative rows with CustomerID: "
    f"{identified_admin_mask.sum():,}"
)
print(
    f"Administrative rows without CustomerID: "
    f"{administrative_purchase_df['CustomerID'].isna().sum():,}"
)

Administrative rows among all valid purchases: 2,310
Administrative rows with CustomerID: 1,542
Administrative rows without CustomerID: 768


In [15]:
before_rows = len(identified_purchase_df)

customer_merchandise_df = (
    identified_purchase_df.loc[
        ~identified_admin_mask
    ]
    .copy()
    .reset_index(drop=True)
)

after_rows = len(customer_merchandise_df)

record_cleaning_step(
    step="C06",
    rule="Exclude reviewed administrative StockCodes from the primary customer merchandise dataset",
    before_rows=before_rows,
    after_rows=after_rows,
    reason=(
        "Fees, postage, samples and manual accounting entries "
        "are not merchandise purchasing behaviour. An inclusive "
        "customer dataset is retained for sensitivity analysis."
    )
)

print(f"Identified purchase rows           : {before_rows:,}")
print(f"Identified administrative rows     : {before_rows - after_rows:,}")
print(f"Customer merchandise rows          : {after_rows:,}")
print(
    f"Customers in merchandise dataset   : "
    f"{customer_merchandise_df['CustomerID'].nunique():,}"
)

Identified purchase rows           : 392,692
Identified administrative rows     : 1,542
Customer merchandise rows          : 391,150
Customers in merchandise dataset   : 4,334


In [16]:
basket_source_df = (
    standardised_purchase_df.loc[
        ~all_purchase_admin_mask
    ]
    .copy()
    .reset_index(drop=True)
)

segment_basket_source_df = (
    basket_source_df.loc[
        basket_source_df["CustomerID"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

segment_basket_source_df["CustomerID"] = (
    segment_basket_source_df["CustomerID"].astype("int64")
)

print(f"Overall merchandise basket rows : {len(basket_source_df):,}")
print(f"Segment-specific basket rows     : {len(segment_basket_source_df):,}")
print(
    f"Overall merchandise invoices    : "
    f"{basket_source_df['InvoiceNo'].nunique():,}"
)
print(
    f"Segment-specific invoices       : "
    f"{segment_basket_source_df['InvoiceNo'].nunique():,}"
)

Overall merchandise basket rows : 522,568
Segment-specific basket rows     : 391,150
Overall merchandise invoices    : 19,773
Segment-specific invoices       : 18,402


In [17]:
preprocessing_audit_df = pd.DataFrame(
    preprocessing_audit
)

display(preprocessing_audit_df)

assert len(standardised_purchase_df) == 524_878
assert len(anonymous_purchase_df) == 132_186
assert len(identified_purchase_df) == 392_692
assert identified_purchase_df["CustomerID"].nunique() == 4_338

assert len(administrative_purchase_df) == 2_310
assert len(customer_merchandise_df) == 391_150
assert customer_merchandise_df["CustomerID"].nunique() == 4_334
assert len(basket_source_df) == 522_568
assert len(segment_basket_source_df) == 391_150

assert customer_merchandise_df["CustomerID"].notna().all()
assert (customer_merchandise_df["Quantity"] > 0).all()
assert (customer_merchandise_df["UnitPrice"] > 0).all()
assert (customer_merchandise_df["LineTotal"] > 0).all()
assert not customer_merchandise_df[
    "StockCode"
].isin(administrative_codes).any()

print("Customer and basket dataset quality checks passed.")

,Step,Rule,Rows_Before,Rows_Removed,Rows_After,Reason
0,C01,Remove additional exact duplicate rows,541909,5268,536641,"Exact duplicates could inflate quantities, rev..."
1,C02,Exclude C-prefixed cancellation invoices,536641,9251,527390,Cancellation invoices do not represent positiv...
2,C03,Retain rows with Quantity greater than zero,527390,1336,526054,Negative quantities represent returns or stock...
3,C04,Retain rows with UnitPrice greater than zero,526054,1176,524878,Zero or negative prices cannot contribute vali...
4,C05,Retain valid purchases with an identified Cust...,524878,132186,392692,Customer segmentation requires a reliable cust...
5,C06,Exclude reviewed administrative StockCodes fro...,392692,1542,391150,"Fees, postage, samples and manual accounting e..."


Customer and basket dataset quality checks passed.


In [18]:
customer_export_df = customer_merchandise_df.copy()
basket_export_df = basket_source_df.copy()
returns_export_df = returns_adjustments_df.copy()
administrative_export_df = administrative_purchase_df.copy()

# Use nullable integer format so CustomerID values are not exported as 17850.0.
basket_export_df["CustomerID"] = (
    basket_export_df["CustomerID"].astype("Int64")
)

returns_export_df["CustomerID"] = (
    returns_export_df["CustomerID"].astype("Int64")
)

administrative_export_df["CustomerID"] = (
    administrative_export_df["CustomerID"].astype("Int64")
)

print("Export datasets prepared.")

Export datasets prepared.


In [19]:
processed_files = {
    "customer_merchandise_transactions": (
        processed_data_path
        / "customer_merchandise_transactions.csv"
    ),
    "merchandise_basket_transactions": (
        processed_data_path
        / "merchandise_basket_transactions.csv"
    ),
    "returns_and_adjustments": (
        processed_data_path
        / "returns_and_adjustments.csv"
    ),
    "administrative_transactions": (
        processed_data_path
        / "administrative_transactions.csv"
    )
}

customer_export_df.to_csv(
    processed_files["customer_merchandise_transactions"],
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

basket_export_df.to_csv(
    processed_files["merchandise_basket_transactions"],
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

returns_export_df.to_csv(
    processed_files["returns_and_adjustments"],
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

administrative_export_df.to_csv(
    processed_files["administrative_transactions"],
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print("Processed datasets saved successfully.")

for dataset_name, file_path in processed_files.items():
    print(f"{dataset_name:<38}: {file_path.name}")

Processed datasets saved successfully.
customer_merchandise_transactions     : customer_merchandise_transactions.csv
merchandise_basket_transactions       : merchandise_basket_transactions.csv
returns_and_adjustments               : returns_and_adjustments.csv
administrative_transactions           : administrative_transactions.csv


In [20]:
preprocessing_audit_df = pd.DataFrame(
    preprocessing_audit
)

preprocessing_audit_df.to_csv(
    project_root
    / "documentation"
    / "preprocessing_audit.csv",
    index=False
)

display(preprocessing_audit_df)

print("Preprocessing audit saved successfully.")

,Step,Rule,Rows_Before,Rows_Removed,Rows_After,Reason
0,C01,Remove additional exact duplicate rows,541909,5268,536641,"Exact duplicates could inflate quantities, rev..."
1,C02,Exclude C-prefixed cancellation invoices,536641,9251,527390,Cancellation invoices do not represent positiv...
2,C03,Retain rows with Quantity greater than zero,527390,1336,526054,Negative quantities represent returns or stock...
3,C04,Retain rows with UnitPrice greater than zero,526054,1176,524878,Zero or negative prices cannot contribute vali...
4,C05,Retain valid purchases with an identified Cust...,524878,132186,392692,Customer segmentation requires a reliable cust...
5,C06,Exclude reviewed administrative StockCodes fro...,392692,1542,391150,"Fees, postage, samples and manual accounting e..."


Preprocessing audit saved successfully.


In [21]:
fingerprint_records = []

for dataset_name, file_path in processed_files.items():
    fingerprint_records.append({
        "Dataset": dataset_name,
        "Filename": file_path.name,
        "Rows": {
            "customer_merchandise_transactions": len(
                customer_export_df
            ),
            "merchandise_basket_transactions": len(
                basket_export_df
            ),
            "returns_and_adjustments": len(
                returns_export_df
            ),
            "administrative_transactions": len(
                administrative_export_df
            )
        }[dataset_name],
        "File_Size_MB": round(
            file_path.stat().st_size / 1024**2,
            2
        ),
        "SHA256": calculate_sha256(file_path)
    })

processed_fingerprints_df = pd.DataFrame(
    fingerprint_records
)

processed_fingerprints_df.to_csv(
    project_root
    / "documentation"
    / "processed_dataset_fingerprints.csv",
    index=False
)

display(processed_fingerprints_df)

,Dataset,Filename,Rows,File_Size_MB,SHA256
0,customer_merchandise_transactions,customer_merchandise_transactions.csv,391150,35.74,32f4aa9cdb2ba48636e2be6467dbafd7a2e04a69daa984...
1,merchandise_basket_transactions,merchandise_basket_transactions.csv,522568,46.99,eab4454dad49c122483dcc07ff67db16a770d2dd53acc0...
2,returns_and_adjustments,returns_and_adjustments.csv,10587,1.01,6f2f566e8c6d67e7a99b3d6e274d235e3efcc9d721ea21...
3,administrative_transactions,administrative_transactions.csv,2310,0.15,9cfd8b4a11a8b0bb49983f22ba972129658767a79b2488...


In [22]:
handover_summary = pd.DataFrame([
    {
        "Dataset": "customer_merchandise_transactions.csv",
        "Rows": len(customer_export_df),
        "Primary_User": "Nirmani K.H.D.T.",
        "Purpose": "Primary input for RFM and additional customer-feature engineering",
        "CustomerID_Required": "Yes",
        "Administrative_Codes": "Excluded"
    },
    {
        "Dataset": "merchandise_basket_transactions.csv",
        "Rows": len(basket_export_df),
        "Primary_User": "Senaratne P.A.R.T.",
        "Purpose": "Overall merchandise basket construction and later segment-specific association mining",
        "CustomerID_Required": "No for overall rules; yes for segment-specific rules",
        "Administrative_Codes": "Excluded"
    },
    {
        "Dataset": "returns_and_adjustments.csv",
        "Rows": len(returns_export_df),
        "Primary_User": "Lakshitha and all members",
        "Purpose": "Separate return, cancellation and adjustment investigation",
        "CustomerID_Required": "No",
        "Administrative_Codes": "Retained where present"
    },
    {
        "Dataset": "administrative_transactions.csv",
        "Rows": len(administrative_export_df),
        "Primary_User": "Nirmani K.H.D.T.",
        "Purpose": "Sensitivity analysis of administrative charges on customer monetary value",
        "CustomerID_Required": "Not always available",
        "Administrative_Codes": "Only administrative codes"
    }
])

handover_summary.to_csv(
    project_root
    / "documentation"
    / "preprocessing_handover.csv",
    index=False
)

display(handover_summary)

,Dataset,Rows,Primary_User,Purpose,CustomerID_Required,Administrative_Codes
0,customer_merchandise_transactions.csv,391150,Nirmani K.H.D.T.,Primary input for RFM and additional customer-...,Yes,Excluded
1,merchandise_basket_transactions.csv,522568,Senaratne P.A.R.T.,Overall merchandise basket construction and la...,No for overall rules; yes for segment-specific...,Excluded
2,returns_and_adjustments.csv,10587,Lakshitha and all members,"Separate return, cancellation and adjustment i...",No,Retained where present
3,administrative_transactions.csv,2310,Nirmani K.H.D.T.,Sensitivity analysis of administrative charges...,Not always available,Only administrative codes


In [23]:
reloaded_customer_df = pd.read_csv(
    processed_files["customer_merchandise_transactions"],
    parse_dates=["InvoiceDate"]
)

reloaded_basket_df = pd.read_csv(
    processed_files["merchandise_basket_transactions"],
    parse_dates=["InvoiceDate"]
)

reloaded_returns_df = pd.read_csv(
    processed_files["returns_and_adjustments"],
    parse_dates=["InvoiceDate"]
)

reloaded_administrative_df = pd.read_csv(
    processed_files["administrative_transactions"],
    parse_dates=["InvoiceDate"]
)

assert len(reloaded_customer_df) == 391_150
assert len(reloaded_basket_df) == 522_568
assert len(reloaded_returns_df) == 10_587
assert len(reloaded_administrative_df) == 2_310

assert reloaded_customer_df["CustomerID"].notna().all()
assert reloaded_customer_df.duplicated().sum() == 0
assert (reloaded_customer_df["Quantity"] > 0).all()
assert (reloaded_customer_df["UnitPrice"] > 0).all()
assert (reloaded_customer_df["LineTotal"] > 0).all()

assert not reloaded_customer_df[
    "StockCode"
].isin(administrative_codes).any()

assert not reloaded_basket_df[
    "StockCode"
].isin(administrative_codes).any()

print("All saved processed datasets passed reload validation.")

All saved processed datasets passed reload validation.


## Preprocessing Conclusion and Handover

The raw dataset was preserved without modification. Exact duplicates were removed from the working data, while cancellations, negative-quantity records and administrative transactions were retained in separate datasets for investigation.

The primary customer-merchandise dataset contains valid, deduplicated, identified-customer purchases with positive quantities and prices. It is ready for RFM and behavioural feature engineering by Nirmani K.H.D.T.

The merchandise basket dataset contains valid merchandise purchase lines and is ready for basket construction by Senaratne P.A.R.T. after final customer segments become available.

No statistical outliers were automatically deleted. Extreme customer behaviour will be handled through transformations, scaling and sensitivity analysis during feature engineering and clustering.

All cleaning decisions, row counts, output purposes and processed-file fingerprints have been documented for reproducibility and group review.